# SMS Spam Classifier — Transfer Learning with DistilBERT
Run each cell **top to bottom**. Each cell = one phase.

| Cell | Phase |
|---|---|
| 1 | Install dependencies |
| 2 | Download dataset |
| 3 | Explore data (EDA) |
| 4 | Tokenise + split 70/15/15 |
| 5 | Fine-tune DistilBERT |
| 6 | Evaluate on validation set |
| 7 | Freeze layers + re-tune |
| 8 | Final test (run once) |
| 9 | Run FastAPI server |

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q transformers torch datasets scikit-learn pandas numpy mlflow fastapi uvicorn python-multipart matplotlib seaborn tqdm accelerate
print('All dependencies installed.')

## Cell 2 — Download dataset

In [ ]:
import urllib.request, zipfile, pandas as pd
from pathlib import Path

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = RAW_DIR / 'spam.csv'

if OUTPUT_CSV.exists():
    print(f'Already exists: {OUTPUT_CSV}')
else:
    UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
    zip_path = RAW_DIR / 'tmp.zip'
    print('Downloading from UCI...')
    urllib.request.urlretrieve(UCI_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        with z.open('SMSSpamCollection') as f:
            lines = f.read().decode('utf-8', errors='replace').splitlines()
    rows = [line.split('\t', 1) for line in lines if '\t' in line]
    df = pd.DataFrame(rows, columns=['label', 'text'])
    df.to_csv(OUTPUT_CSV, index=False)
    zip_path.unlink()
    print(f'Saved {len(df)} rows to {OUTPUT_CSV}')

df = pd.read_csv(OUTPUT_CSV)
print(df.head())

## Cell 3 — Explore data (EDA)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

df = pd.read_csv('data/raw/spam.csv')
# Handle both Kaggle (v1/v2) and UCI (label/text) column names
if 'v1' in df.columns:
    df = df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})
df['label'] = df['label'].str.strip().str.lower()
df['length'] = df['text'].str.len()

print('=== Shape:', df.shape)
print(df['label'].value_counts())
print('\n=== Text length stats:')
print(df.groupby('label')['length'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['label'].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette=['steelblue','salmon'], ax=axes[0])
axes[0].set_title('Class distribution')
for cls, grp in df.groupby('label'):
    grp['length'].plot.hist(bins=50, alpha=0.6, label=cls, ax=axes[1])
axes[1].set_title('Message length by class')
axes[1].legend()
plt.tight_layout()
plt.show()

## Cell 4 — Tokenise + 70/15/15 split

In [ ]:
import json, pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

PROCESSED_DIR = Path('data/processed')
CHECKPOINT = 'distilbert-base-uncased'
MAX_LENGTH = 128

df = pd.read_csv('data/raw/spam.csv')
if 'v1' in df.columns:
    df = df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})
df = df[['label','text']].dropna()
df['label'] = df['label'].str.strip().str.lower()
df['labels'] = (df['label'] == 'spam').astype(int)

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df['labels'], random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['labels'], random_state=42)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

raw_ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text','labels']], preserve_index=False),
    'val':   Dataset.from_pandas(val_df[['text','labels']],   preserve_index=False),
    'test':  Dataset.from_pandas(test_df[['text','labels']],  preserve_index=False),
})

print(f'Loading tokenizer: {CHECKPOINT}...')
tokenizer = DistilBertTokenizerFast.from_pretrained(CHECKPOINT)

def tokenise(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

tokenised_ds = raw_ds.map(tokenise, batched=True)
tokenised_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
tokenised_ds.save_to_disk(str(PROCESSED_DIR))
Path(PROCESSED_DIR / 'label_map.json').write_text(json.dumps({'0':'ham','1':'spam'}))
print(f'Tokenised dataset saved to {PROCESSED_DIR}')

## Cell 5 — Fine-tune DistilBERT (≈10 min on CPU, ≈90 sec on GPU)

In [ ]:
import numpy as np, mlflow
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

CHECKPOINT = 'distilbert-base-uncased'
BEST_DIR   = Path('models/best')
BEST_DIR.mkdir(parents=True, exist_ok=True)

ds = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])

model     = DistilBertForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2)
tokenizer = DistilBertTokenizerFast.from_pretrained(CHECKPOINT)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds)}

args = TrainingArguments(
    output_dir='models/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['val'],
                  tokenizer=tokenizer, compute_metrics=compute_metrics)

mlflow.set_experiment('sms-spam-distilbert')
with mlflow.start_run(run_name='full-fine-tune'):
    mlflow.log_params({'epochs':3,'lr':2e-5,'batch':32,'frozen':'none'})
    result = trainer.train()
    val_m  = trainer.evaluate(ds['val'])
    mlflow.log_metrics({'val_accuracy': val_m['eval_accuracy'], 'val_f1': val_m['eval_f1']})
    trainer.save_model(str(BEST_DIR))
    tokenizer.save_pretrained(str(BEST_DIR))

print(f"\nDone. Val accuracy: {val_m['eval_accuracy']:.4f}  F1: {val_m['eval_f1']:.4f}")
print(f'Model saved to {BEST_DIR}')

## Cell 6 — Evaluate on validation set + confusion matrix

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datasets import load_from_disk
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained('models/best')
tokenizer = DistilBertTokenizerFast.from_pretrained('models/best')

trainer = Trainer(model=model, tokenizer=tokenizer,
                  args=TrainingArguments(output_dir='models/eval_tmp',
                                        per_device_eval_batch_size=64,
                                        report_to='none'))
preds  = trainer.predict(ds['val'])
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=['ham','spam']))

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['ham','spam'], yticklabels=['ham','spam'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Validation')
plt.tight_layout(); plt.show()

## Cell 7 — Freeze 5 layers + re-tune (only 3.5% of params)

In [ ]:
import numpy as np, mlflow
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

FROZEN_DIR = Path('models/frozen')
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained('models/best')
tokenizer = DistilBertTokenizerFast.from_pretrained('models/best')

# Freeze embeddings + first 5 transformer layers
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False
for i in range(5):
    for param in model.distilbert.transformer.layer[i].parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds)}

args = TrainingArguments(
    output_dir='models/frozen_ckpt',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
    seed=42,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['val'],
                  tokenizer=tokenizer, compute_metrics=compute_metrics)

mlflow.set_experiment('sms-spam-distilbert')
with mlflow.start_run(run_name='frozen-5-layers'):
    mlflow.log_params({'epochs':3,'lr':3e-5,'batch':32,'frozen_layers':5})
    trainer.train()
    val_m = trainer.evaluate(ds['val'])
    mlflow.log_metrics({'val_accuracy': val_m['eval_accuracy'], 'val_f1': val_m['eval_f1']})
    trainer.save_model(str(FROZEN_DIR))
    tokenizer.save_pretrained(str(FROZEN_DIR))

print(f"\nFrozen model — Val accuracy: {val_m['eval_accuracy']:.4f}  F1: {val_m['eval_f1']:.4f}")

## Cell 8 — Final test evaluation (run this ONCE)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

# Use frozen model if available, else best
model_path = 'models/frozen' if Path('models/frozen').exists() else 'models/best'
print(f'Loading model from: {model_path}')

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

trainer = Trainer(model=model, tokenizer=tokenizer,
                  args=TrainingArguments(output_dir='models/test_tmp',
                                        per_device_eval_batch_size=64,
                                        report_to='none'))
preds  = trainer.predict(ds['test'])
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print('\n========== FINAL TEST RESULTS ==========')
print(classification_report(y_true, y_pred, target_names=['ham','spam']))
print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'F1 (spam): {f1_score(y_true, y_pred):.4f}')
print('=========================================')

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Oranges',
            xticklabels=['ham','spam'], yticklabels=['ham','spam'], ax=ax)
ax.set_title('Confusion Matrix — Test Set (Final)')
plt.tight_layout(); plt.show()

## Cell 9 — Quick inference (test any message)

In [ ]:
import torch
from pathlib import Path
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_path = 'models/frozen' if Path('models/frozen').exists() else 'models/best'
model     = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model.eval()

def predict(text: str) -> dict:
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0]
    label = 'spam' if probs[1] > probs[0] else 'ham'
    return {'label': label, 'confidence': f'{probs.max().item():.2%}'}

# Try your own messages here!
test_messages = [
    'Congratulations! You won a FREE iPhone. Click now to claim!',
    'Hey, are we still on for dinner tonight?',
    'URGENT: Your account has been suspended. Verify now at http://scam.com',
    'Can you pick up some milk on your way home?',
]

for msg in test_messages:
    result = predict(msg)
    print(f"[{result['label'].upper():4s}] {result['confidence']}  →  {msg[:60]}")